# Modelos Multimodales
### Máster en IA y Computación Cuántica Aplicada a los Mercados Financieros (MIAX)

---

- **Autor:** Pablo Hernández Cámara
- **Contacto:** [pablo.hernandez-camara@uv.es](mailto:pablo.hernandez-camara@uv.es)
- **Notebook:** Transcripción de audio con Whisper

"En este notebook vamos a pasar de **audio a texto** utilizando Whisper, un modelo de reconocimiento automático del habla (*Automatic Speech Recognition, ASR*). Este notebook conecta especialmente bien con el anterior: podemos generar audio con un modelo y, a continuación, utilizar otro modelo para **escuchar ese audio y convertirlo de nuevo en texto**.

Utilizaremos Hugging Face `transformers` y el ecosistema de audio necesario para preparar y procesar la señal.

In [1]:
!pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 12.0 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5


# Cargar Whisper

Whisper es un modelo de **reconocimiento automático del habla (ASR)**. El flujo conceptual es: **audio → procesamiento de la señal → Whisper → texto**

Vamos a utilizar `openai/whisper-base`, una versión relativamente ligera del modelo. La GPU no es estrictamente necesaria para entender el ejemplo, pero permite acelerar considerablemente la inferencia.

In [2]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = AutoModelForSpeechSeq2Seq.from_pretrained("openai/whisper-base", torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True)
model.to(device)

processor = AutoProcessor.from_pretrained("openai/whisper-base")

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  290MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.81k [00:00<?, ?B/s]

¿Qué acabamos de cargar?

- **`model`**: la red neuronal que realiza la transcripción.
- **`processor`**: prepara el audio y gestiona la representación que necesita el modelo.
- **`device`**: indica si ejecutaremos la inferencia en CPU o GPU.
- **`torch_dtype`**: usamos `float16` en GPU para reducir memoria y acelerar el cálculo cuando es compatible.

Pero vamos a usar la `pipeline`, que nos permite encapsular buena parte de este procesamiento y proporciona una interfaz sencilla para ejecutar una tarea de reconocimiento automático del habla.

In [ ]:
pipe = pipeline(model="openai/whisper-base")

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Cargamos el audio que hemos generado previamente, pero podria ser cualquiera, el mismo procedimiento se puede aplicar a otros archivos de audio compatibles.

In [ ]:
from IPython.display import Audio
audio = Audio(filename="descarga.wav")

audio

Un archivo de audio no es simplemente "texto": contiene una **señal** representada mediante números.

Aquí utilizamos `scipy` para leer el WAV y obtener:
- `samplerate`: frecuencia de muestreo, es decir, cuántas muestras se toman por segundo.
- `data`: las muestras que representan la señal de audio.

Primero inspeccionaremos estos valores para entender qué estamos pasando al modelo.

In [ ]:
from scipy.io import wavfile
import numpy as np
samplerate, data = wavfile.read('descarga.wav')

In [ ]:
print(f"Frecuencia de muestreo: {samplerate} Hz")
print(f"Forma del array: {data.shape}")
print(f"Tipo de dato original: {data.dtype}")

In [ ]:
print(f"Primeros 10 valores: {data[:10]}")

array([-196, -169,  -67, ...,  324,  338,  397], dtype=int16)

### Normalización de la señal

En un WAV PCM de 16 bits, las muestras suelen estar representadas mediante enteros en un rango aproximado de `-32768` a `32767`. Las convertimos a `float32` y las escalamos para trabajar con valores normalizados:

`valor_normalizado = valor_original / 32768`

Esto es un ejemplo sencillo de **preprocesado de datos** antes de la inferencia.


In [ ]:
data = data.astype(np.float32)/32768.0

Ahora tenemos todos los elementos: **archivo WAV → señal numérica preprocesada → pipeline de Whisper → texto**

Ejecutamos la inferencia y observamos el resultado:

In [ ]:
result = pipe(data)

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece token

In [ ]:
result['text']

' He generado una huye de forma muy sencilla para la clase de Pablo'

Prueba con un audio diferente y observa cómo cambia la transcripción. ¿Dónde aparecen errores? ¿Influye el ruido?